***

## Preparing Workspace

***

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:

# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'DOF')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'DOF Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'DOF')
    path_config  = os.path.join(path_code, 'config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'DOF')
    path_config  = os.path.join(path_code, 'config')
    

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

## Preparing Import Parameters

***

In [ ]:

## Population Estimates
## E4
# 2020-2024
# 2010-2020
# 2000-2010
url_2020_2024 = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
url_2010_2020 = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
url_2000_2010 = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"

## Population and Housing Estimates
## E5
# 2020-2024
url_2020_2024 = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
## E8
# 2010-2020
# 2000-2010
url_2010_2020 = "https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
url_2000_2010 = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"


### Population Estimates

E4

- 2020 to 2024 -> https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx
- 2010 to 2020 -> https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx
- 2000 to 2010 -> https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx

### Population and Housing Estimates

E5

- 2020 to 2024 -> https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx


E8

- 2010 to 2020 -> https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx
- 2000 to 2010 -> https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx


These urls listed above can be copied and pasted into your browser, so you can see the exact table we are importing.  The tables (and url) can also be found here https://dof.ca.gov/forecasting/demographics/.  Click on "Estimates" to access tables.


NOTE -  _Sometimes, the tables are adjusted (even older tables).  The code below makes certain assumptions about the tables (like how many rows we need to skip to get the column headers correct).  If the code below errors out, you will need to manually check each tables at the links listed above to make sure that the tables are in the format as coded below._

In [ ]:

## Import E5 and E8 DOF data ##

# Set custom browser user agent so that DOF doesn't deny your request to access data
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

# Organize E5/E8 urls by year
dict_url = {
    2020:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
    , 2010:"https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
    , 2000:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"
}


# Import Area Codes
df_fips = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS')
df_fips = df_fips[df_fips['State'] == 'CA'].dropna()

df_fips['County FIPS'] = df_fips['County FIPS'].astype(str).apply('{:0>3}'.format)
df_fips['State FIPS' ] = df_fips['State FIPS' ].astype(str).apply('{:0>2}'.format)

SACOG_counties = df_fips[df_fips['MPO'] == 'SACOG']['County Name'].values

print(SACOG_counties)
df_fips.head()


***

## Importing

***

_Import code was last updated on Aug 7, 2024_

In [ ]:

# Create empty lists to store pulled data
list_df_state    = []
list_df_counties = []
list_df_cities   = []
list_df_balance  = []


# Import 1 decade at a time
# 2000-2010 data is organized differently than 2010-2024
for year in list(dict_url.keys()):
    if year == 2000:

        # Pull data using url and custom user agent
        # Drop missings, convert date field, create year field
        # Fill in county and city fields (the excel data sheet just has missings where it should be city/county)
        # Separate out and roll up population/household totals by city/county/balance
        # San francisco is special case that needs to be manually adjusted
        
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        df = df.dropna(subset = ['Household'])
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        
        # Initialize 'County' and 'City' columns
        df['County'] = np.nan
        df['City'  ] = np.nan
        
        county_temp = None
        
        # Reset index
        df.reset_index(drop=True, inplace=True)
        
        for i in range(len(df)):
            if pd.notnull(df.loc[i, 'County / City']):
                if county_temp is not None:
                    df.loc[i, 'County'] = county_temp
                    df.loc[i, 'City'] = df.loc[i, 'County / City']
                    county_temp = None
                else:
                    county_temp = df.loc[i, 'County / City']
        
        df['County'].fillna(method='ffill', inplace = True)
        df['City'  ].fillna(method='ffill', inplace = True)
        
        # shift up and then fill the last row
        df['County'] = df['County'].shift(-1)
        df['City'  ] = df['City'  ].shift(-1)
        
        df['County'].fillna(method = 'ffill', inplace = True)
        df['City'  ].fillna(method = 'ffill', inplace = True)
        df = df.drop(columns = ['County / City'])

        df = df[df['Year'] != 2010]

        df = df.rename(columns = {'Household': 'Household Population'
                                  , 'Total'  : 'Population'
                                  , 'Total.1': 'Housing Units'})
        
        df['Unoccupied'] = df['Housing Units'] - df['Occupied']
        df = df[['County', 'City', 'Date', 'Year', 'Population', 'Household Population', 'Group Quarters', 
                                  'Housing Units', 'Single'    , 'Multiple'            , 'Mobile Homes'  , 'Occupied', 'Unoccupied',
                                   'Vacancy Rate', 'Persons Per Household']].rename(columns = {'Persons Per Household':'Persons per Household'})
        
        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        df = df.merge(df_fips[['County Name', 'MPO']], left_on = 'County', right_on = 'County Name', how = 'left').drop('County Name', axis = 1)
        df.loc[df['MPO'].isna(), 'MPO'] = 'Rest of CA'
        
        # Subset
        df_sf = df[df['County'] == 'San Francisco']
        df_sf.loc[:, 'City'] = 'County Total'
        df = pd.concat([df, df_sf])
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop = True)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )

    else:

        # Pull data using url and custom user agent
        # Drop missings, convert date field, create year field
        # Separate out and roll up population/household totals by city/county/balance
        # San francisco is special case that needs to be manually adjusted
        
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        if year == 2020:
            df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        if year == 2010:
            df = pd.read_excel(request, sheet_name = 1, skiprows = 1, engine='openpyxl')
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        if year == 2010:
            df = df[df['Year'] != 2020]

        df = df.rename(columns = {'Household': 'Household Population'
                           , 'Total'  : 'Population'
                           , 'Total.1': 'Housing Units'})
        if year == 2010:
            cols_to_int = ['Population', 'Household Population', 'Group Quarters', 'Housing Units', 'Single Detached', 'Single Attached',
                           'Two to Four', 'Five Plus', 'Mobile Homes', 'Occupied', 'Vacancy Rate', 'Persons per Household']
            df[cols_to_int] = df[cols_to_int].apply(pd.to_numeric, errors='coerce')
            df = df.dropna()
        
        df['Unoccupied'] = df['Housing Units'] - df['Occupied']
        df['Single'  ] = df['Single Attached'] + df['Single Detached']
        df['Multiple'] = df['Two to Four'    ] + df['Five Plus'      ]
        
        df = df[['County', 'City', 'Date', 'Year', 'Population', 'Household Population', 'Group Quarters', 
                                  'Housing Units', 'Single'    , 'Single Detached'     , 'Single Attached', 'Multiple',
                                    'Two to Four', 'Five Plus' , 'Mobile Homes'        ,
                                       'Occupied', 'Unoccupied', 'Vacancy Rate'        , 'Persons per Household']]
        
        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        df = df.merge(df_fips[['County Name', 'MPO']], left_on = 'County', right_on = 'County Name', how = 'left').drop('County Name', axis = 1)
        df.loc[df['MPO'].isna(), 'MPO'] = 'Rest of CA'

        # Subset
        if year == 2020:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'San Francisco'
            df = pd.concat([df, df_sf])

        if year == 2010:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'County Total'
            df = pd.concat([df, df_sf])
            
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop = True)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )
            

df_state    = pd.concat(list_df_state   )
df_counties = pd.concat(list_df_counties)
df_cities   = pd.concat(list_df_cities  )
df_balance  = pd.concat(list_df_balance )

df_state    = df_state   .sort_values(['City'  , 'Year'], ascending = [True, False]).rename(columns = {'City':'Jurisdiction'})
df_counties = df_counties.sort_values(['County', 'Year'], ascending = [True, False])
df_cities   = df_cities  .sort_values(['City'  , 'Year'], ascending = [True, False]).rename(columns = {'City':'Jurisdiction'})
df_balance  = df_balance .sort_values(['County', 'Year'], ascending = [True, False])


df_counties = df_counties.set_index(['MPO', 'County'                ]).reset_index()
df_cities   = df_cities  .set_index(['MPO', 'County', 'Jurisdiction']).reset_index()
df_balance  = df_balance .set_index(['MPO', 'County'                ]).reset_index()


print('By State'   )
print(df_state   .head())
print('By Counties')
print(df_counties.head())
print('By Cities'  )
print(df_cities  .head())
print('By Balance' )
print(df_balance .head())

# Mariposa, Alpine, Trinity not included in the "Cities" df because they don't have any jurisdictions

In [ ]:

# Export raw data in full
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics')

# All of CA
df_cities2  = df_cities .copy()
df_balance2 = df_balance.copy()

df_balance2['Jurisdiction'] = 'Unincorporated'
df_cities3 = pd.concat([df_cities2, df_balance2])

df_cities3 = df_cities3.sort_values(['County', 'Jurisdiction', 'Year'], ascending = [True, True, False])

# df_cities3.to_excel(os.path.join(path_out, 'DOF_E5_and_E8_Jurisdictions.xlsx'), index = False)
df_cities3.head()


***

## Pop_1

***

In [ ]:

indicator_name = 'Pop_1'

# Import about table
df_about = pd.read_excel(os.path.join(path_config, 'DOF Configuration File.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
print(folder)

path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', indicator_name + ' ' + folder)
path_csv = os.path.join(path_agol, indicator_name)

df_pop1 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop1 = df_pop1[['County', 'Year', 'Population', 'Household Population']].drop_duplicates()

# # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Counties.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop1.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop1.to_csv(os.path.join(path_csv, indicator_name + '_1_DOF_Counties.csv'), index = False)

# View
df_pop1.head()

In [ ]:
# path_plots = os.path.join(path_agol, 'plots')

df_plot = df_pop1.copy()


x = 'Year'
y = 'Population'
color = 'County'
labels = 'County'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = True
                 , labels = labels
                )

fig.update_layout(title = 'Population by County')

# fig.write_html(
#     os.path.join(
#         path_plots
#         , ''.join(['Pop_1' + '_'
#                    , 'Population by County_'
#                    , 'line_'
#                    , '.html'])
#     )
# )
    

fig.show()

***

## Pop_2

***

In [ ]:
df_cities.head()

In [ ]:
df_balance.head()

In [ ]:
indicator_name = 'Pop_2'

# Import about table
df_about = pd.read_excel(os.path.join(path_config, 'DOF Configuration File.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
print(folder)

path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', indicator_name + ' ' + folder)
path_csv = os.path.join(path_agol, indicator_name)

# Jurisdiction
df_pop2 = df_cities[df_cities['County'].isin(SACOG_counties)]
df_pop2_balance = df_balance[df_balance['County'].isin(SACOG_counties)]
df_pop2_balance['Jurisdiction'] = 'Unincorporated'

df_pop2 = pd.concat([df_pop2, df_pop2_balance])  
df_pop2 = df_pop2[['County', 'Jurisdiction', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['Jurisdiction'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['Jurisdiction', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()*100
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()*100
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['County', 'Jurisdiction', 'Year'], ascending = [True, True, False])

# # Export
# # writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Jurisdictions.xlsx'), engine = 'xlsxwriter')
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Jurisdictions.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop2.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop2.to_csv(os.path.join(path_csv, indicator_name + '_DOF_Jurisdictions.csv'), index = False)


# County
df_pop2 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop2 = df_pop2[['County', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['County'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()*100
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()*100
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, False])

# # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Counties.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop2.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop2.to_csv(os.path.join(path_csv, indicator_name + '_DOF_Counties.csv'), index = False)

# 5 year time lapses
df_pop2 = df_counties[df_counties['County'].isin(SACOG_counties)]
df_pop2 = df_pop2[df_pop2['Year'].isin([2000, 2005, 2010, 2015, 2020, 2024])]
df_pop2 = df_pop2[['County', 'Year', 'Population']]
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, True])
df_pop2['Population_Diff'] = df_pop2.groupby('County')['Population'].diff()
df_pop2['Year_Diff'] = df_pop2.groupby('County')['Year'].diff()
df_pop2['Year_Diff'].fillna(1, inplace=True)  # Assuming the difference of one year when NaN, adjust if needed
df_pop2['Population_Diff'].fillna(0, inplace=True)

df_pop2['Previous_POP'] = df_pop2.groupby('County')['Population'].shift(1)
df_pop2['Result'] = ((df_pop2['Population_Diff']) / df_pop2['Previous_POP']) * 100 / df_pop2['Year_Diff']

df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, False])


# Reshape data to wide format
df_pop2 = df_pop2.pivot_table(index = ['County']
                         , columns = 'Year'
                         , values = 'Result').reset_index()

df_pop2.columns = ['County', '2000-2005', '2005-2010', '2010-2015', '2015-2020', '2020-2024']

# # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Counties_5 Year Time Series.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop2.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop2.to_csv(os.path.join(path_csv, indicator_name + '_DOF_Counties_5 Year Time Series.csv'), index = False)


# View
df_pop2.head()

In [ ]:
# path_plots = os.path.join(path_out, 'plots')

df_plot = df_pop2.copy()
df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)


x = 'Year'
y = 'Population_GR'
color = 'County'
labels = 'County'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = True
                 , labels = labels
                )

fig.add_hline(y = 0, line_dash = 'dash', line_color = 'black')

fig.update_layout(title = 'Population Growth Rate by County (%)')

# fig.write_html(
#     os.path.join(
#         path_plots
#         , ''.join(['Pop_2' + '_'
#                    , 'Population Growth Rate by County_'
#                    , 'line_'
#                    , '.html'])
#     )
# )
    

fig.show()

***

## Pop_5

***

In [ ]:
df_cities.head()

In [ ]:
indicator_name = 'Pop_5'

# Import about table
df_about = pd.read_excel(os.path.join(path_config, 'DOF Configuration File.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
print(folder)

path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', indicator_name + ' ' + folder)
path_csv = os.path.join(path_agol, indicator_name)


# Jurisdiction
df_pop5 = df_cities[df_cities['County'].isin(SACOG_counties)]
df_pop5_balance = df_balance[df_balance['County'].isin(SACOG_counties)]
df_pop5_balance['Jurisdiction'] = 'Unincorporated'
df_pop5 = pd.concat([df_pop5, df_pop5_balance])  

df_pop5 = df_pop5[['MPO', 'County', 'Jurisdiction', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5[df_pop5['Jurisdiction'] != 'Incorporated']
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Jurisdiction', 'Year'], ascending = [True, True, True, False])


# # # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Jurisdictions.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop5.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop5.to_csv(os.path.join(path_csv, indicator_name + '_DOF_Jurisdiction.csv'), index = False)

# County
df_pop5 = df_counties[['MPO', 'County', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5[df_pop5['County'] != 'Incorporated']
df_pop5 = df_pop5.sort_values(['County', 'Year'], ascending = [True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'County', 'Year'], ascending = [True, True, False])


# # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_Counties.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop5.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop5.to_csv(os.path.join(path_csv, indicator_name + '_DOF_County.csv'), index = False)


# MPO
df_pop5 = df_counties[df_counties['County'] != 'Incorporated']
df_pop5 = df_pop5[['MPO', 'Year', 'Population', 'Household Population']]
df_pop5 = df_pop5.groupby(['MPO', 'Year'], as_index = False).agg(sum)
df_pop5 = df_pop5.sort_values(['MPO', 'Year'], ascending = [True, True])
df_pop5['Population_GR'          ] = df_pop5['Population'          ].pct_change()
df_pop5['Household Population_GR'] = df_pop5['Household Population'].pct_change()
df_pop5.loc[df_pop5['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop5.loc[df_pop5['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop5.loc[df_pop5['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop5 = df_pop5.sort_values(['MPO', 'Year'], ascending = [True, False])


# # Export
# writer = pd.ExcelWriter(os.path.join(path_out, indicator_name + '_DOF_MPO.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# df_pop5.to_excel(writer, sheet_name = 'Data', index = False)
# writer.close()
# df_pop5.to_csv(os.path.join(path_csv, indicator_name + '_DOF_MPO.csv'), index = False)


# View
df_pop5.head()

In [ ]:
path_plots = os.path.join(path_out, 'plots')

# df_plot = df_pop5.copy()
# df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)


# x = 'Year'
# y = 'Population_GR'
# color = 'MPO'
# line_dash = 'County'
# # labels = 'County'


# fig = px.line(df_plot
#                  , x = x
#                  , y = y
#                  , color = color
#                  , line_dash = line_dash
#                  , markers = True
#                  # , labels = labels
#                 )

# fig.update_layout(title = 'Population Growth Rate by County and by MPO (%)')

# # fig.write_html(
# #     os.path.join(
# #         path_plots
# #         , ''.join(['Pop_5' + '_'
# #                    , 'Population Growth Rate by County and by MPO_'
# #                    , 'line_'
# #                    , '.html'])
# #     )
# # )
    

# fig.show()


## Just by MPO ##
df_plot = df_pop5.copy()
df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)


x = 'Year'
y = 'Population_GR'
color = 'MPO'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 , markers = True
                )

fig.add_hline(y = 0, line_dash = 'dash', line_color = 'black')
fig.update_layout(title = 'Population Growth Rate by MPO (%)')

# fig.write_html(
#     os.path.join(
#         path_plots
#         , ''.join(['Pop_5' + '_'
#                    , 'Population Growth Rate by MPO_'
#                    , 'line_'
#                    , '.html'])
#     )
# )
    

fig.show()



***

## Cost_3

***

In [ ]:
# path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_3')
# path_csv = os.path.join(path_agol, 'Cost_3')

# # Juridiction
# df_cost3 = df_cities[df_cities['County'].isin(SACOG_counties)]
# df_cost3_balance = df_balance[df_balance['County'].isin(SACOG_counties)]
# df_cost3_balance['Jurisdiction'] = 'Unincorporated'
# df_cost3 = pd.concat([df_cost3, df_cost3_balance])  

# df_cost3 = df_cost3[['Jurisdiction', 'Year', 'Housing Units', 'Occupied', 'Unoccupied']]
# df_cost3 = df_cost3.sort_values(['Jurisdiction', 'Year'], ascending = [True, True])
# df_cost3['Housing Units_GR'] = df_cost3['Housing Units'].pct_change()
# df_cost3['Occupied_GR'     ] = df_cost3['Occupied'     ].pct_change()
# df_cost3['Unoccupied_GR'   ] = df_cost3['Unoccupied'   ].pct_change()
# df_cost3.loc[df_cost3['Year'] == 2000, 'Housing Units_GR'] = np.nan
# df_cost3.loc[df_cost3['Year'] == 2000, 'Occupied_GR'     ] = np.nan
# df_cost3.loc[df_cost3['Year'] == 2000, 'Unoccupied_GR'   ] = np.nan
# df_cost3 = df_cost3.sort_values(['Jurisdiction', 'Year'], ascending = [True, False])


# # # Export
# # writer = pd.ExcelWriter(os.path.join(path_out, 'Cost_3_DOF_Jurisdiction.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# # df_cost3.to_excel(writer, sheet_name = 'Data', index = False)
# # writer.close()
# # df_cost3.to_csv(os.path.join(path_csv, 'Cost_3_DOF_Jurisdiction.csv'), index = False)


# # County
# df_cost3 = df_counties[df_counties['County'].isin(SACOG_counties)]

# df_cost3 = df_cost3[['County', 'Year', 'Housing Units', 'Occupied', 'Unoccupied']]
# df_cost3 = df_cost3.sort_values(['County', 'Year'], ascending = [True, True])
# df_cost3['Housing Units_GR'] = df_cost3['Housing Units'].pct_change()
# df_cost3['Occupied_GR'     ] = df_cost3['Occupied'     ].pct_change()
# df_cost3['Unoccupied_GR'   ] = df_cost3['Unoccupied'   ].pct_change()
# df_cost3.loc[df_cost3['Year'] == 2000, 'Housing Units_GR'] = np.nan
# df_cost3.loc[df_cost3['Year'] == 2000, 'Occupied_GR'     ] = np.nan
# df_cost3.loc[df_cost3['Year'] == 2000, 'Unoccupied_GR'   ] = np.nan
# df_cost3 = df_cost3.sort_values(['County', 'Year'], ascending = [True, False])

# # # Export
# # writer = pd.ExcelWriter(os.path.join(path_out, 'Cost_3_DOF_County.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace')
# # df_cost3.to_excel(writer, sheet_name = 'Data', index = False)
# # writer.close()
# # df_cost3.to_csv(os.path.join(path_csv, 'Cost_3_DOF_County.csv'), index = False)


# # View
# df_cost3.head()

In [ ]:
# # Cost_3
# path_plots = os.path.join(path_out, 'plots')

# df_plot = df_cost3.copy()
# df_plot['Unoccupied_GR'] = round(df_plot['Unoccupied_GR']*100, 1)
# df_plot = df_plot[df_plot['County'] != 'San Benito'] # i think there is a data quality issue with san benito

# x = 'Year'
# y = 'Unoccupied_GR'
# color = 'County'
# labels = 'County'


# fig = px.line(df_plot
#                  , x = x
#                  , y = y
#                  , color = color
#                  # , line_dash = line_dash
#                  , markers = True
#                  , labels = labels
#                 )

# fig.add_hline(y = 0, line_dash = 'dash', line_color = 'black')
# fig.update_layout(title = 'Unoccupied Housing Growth Rate by County (%)')

# # fig.write_html(
# #     os.path.join(
# #         path_plots
# #         , ''.join(['Cost_3' + '_'
# #                    , 'Unoccupied Housing Growth Growth Rate by County_'
# #                    , 'line_'
# #                    , '.html'])
# #     )
# # )

# fig.show()